# BrainDecoding CPU 特征缓存预热与云端单文件 Tar 归档打包

本笔记本在 **Google Colab CPU** 实例上完成离线预处理，并将标准化派生缓存打包为单一 `.tar` 归档存入 Google Drive `archives/`，彻底消除 GPU 训练期间读取大量散碎文件的 Drive FUSE 瓶颈：

- `broderick2018`（默认） → `archives/features_broderick2018.tar`（0.1–40 Hz EEG 50 Hz 重采样信号 + T5-large 第 12 层词向量 + 19 被试事件表与 `N=50` 词表）
- `pallier2025` → `archives/features_meg.tar`
- `chineseeeg2_littleprince` → `archives/features_eeg.tar`

推荐设置 `STAGE_RAW_TO_LOCAL_SSD = True`：先将原始数据同步到 Colab 本地高速盘 `/content/raw_stage` 并通过 `/content/studies_links` 软链接桥接，再执行缓存构建与 `.tar` 打包。
        

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

DATASET = "broderick2018"  # @param ["broderick2018", "pallier2025", "chineseeeg2_littleprince"]
REPO_DIR = "/content/BrainDecoding"  # @param {type:"string"}
REPO_GIT_URL = "https://github.com/wdongdong0708-jpg/BrainDecoding2.git"  # @param {type:"string"}
REPO_GIT_BRANCH = "Broderick-EEG"  # @param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/BrainDecoding"  # @param {type:"string"}
LOCAL_CACHE_DIR = "/content/local_cache"  # @param {type:"string"}
LINKS_DIR = "/content/studies_links"  # @param {type:"string"}
STAGE_RAW_TO_LOCAL_SSD = True  # @param {type:"boolean"}
INCLUDE_TEST = True  # @param {type:"boolean"}
FORCE_REBUILD = False  # @param {type:"boolean"}
SUBJECTS = ""  # @param {type:"string"}
NUM_SUBJECTS = 0  # @param {type:"integer"}

DATASET_RAW_SUBDIR = {
    "broderick2018": "Broderick2018_NaturalSpeech",
    "pallier2025": "pallier2025",
    "chineseeeg2_littleprince": "ChineseEEG-2",
}
DATASET_ARCHIVE_NAME = {
    "broderick2018": "features_broderick2018.tar",
    "pallier2025": "features_meg.tar",
    "chineseeeg2_littleprince": "features_eeg.tar",
}
if DATASET not in DATASET_RAW_SUBDIR:
    raise ValueError(f"不支持的数据集：{DATASET}")

DRIVE_BASE = Path(DRIVE_ROOT)
DRIVE_DATA_DIR = DRIVE_BASE / "data"
DRIVE_ARCHIVES_DIR = DRIVE_BASE / "archives"
DRIVE_ARCHIVES_DIR.mkdir(parents=True, exist_ok=True)

repo_path = Path(REPO_DIR)
if not repo_path.is_dir():
    drive_repo = DRIVE_BASE / "BrainDecoding"
    if (drive_repo / "pyproject.toml").is_file():
        repo_path = drive_repo
    elif REPO_GIT_URL.strip():
        subprocess.check_call(
            ["git", "clone", "-b", REPO_GIT_BRANCH, REPO_GIT_URL.strip(), str(repo_path)]
        )
    else:
        raise FileNotFoundError(
            f"未找到项目仓库 {repo_path}（也未在 {drive_repo} 找到），请上传代码或填入 REPO_GIT_URL。"
        )

REPO_PATH = repo_path
DERIVED_CACHE_DIR = Path(LOCAL_CACHE_DIR) / "derived"
RAW_STAGE_ROOT = Path("/content/raw_stage")
ARCHIVE_PATH = DRIVE_ARCHIVES_DIR / DATASET_ARCHIVE_NAME[DATASET]

disk_local = shutil.disk_usage("/content")
disk_drive = shutil.disk_usage(str(DRIVE_BASE))
print(f"仓库路径: {REPO_PATH}")
print(f"目标数据集: {DATASET} -> {ARCHIVE_PATH.name}")
print(f"本地 SSD 可用: {disk_local.free / (1024**3):.2f} GB | Drive 可用: {disk_drive.free / (1024**3):.2f} GB")
        

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

VENV_DIR = Path("/content/proj-venv")
setup_script = REPO_PATH / "scripts/colab_setup.py"
setup_cmd = [
    sys.executable,
    str(setup_script),
    "--venv-dir",
    str(VENV_DIR),
    "--project-root",
    str(REPO_PATH),
    "--drive-data-dir",
    str(DRIVE_DATA_DIR),
    "--links-dir",
    LINKS_DIR,
    "--local-cache-dir",
    LOCAL_CACHE_DIR,
]
print("执行 Colab 环境初始化与软链接桥接...")
subprocess.check_call(setup_cmd, cwd=str(REPO_PATH))

VENV_PYTHON = VENV_DIR / "bin/python"
if not VENV_PYTHON.is_file():
    VENV_PYTHON = Path(sys.executable)

raw_folder = DATASET_RAW_SUBDIR[DATASET]
drive_raw_src = DRIVE_DATA_DIR / raw_folder
link_target = Path(LINKS_DIR) / raw_folder

if STAGE_RAW_TO_LOCAL_SSD:
    if not drive_raw_src.exists():
        raise FileNotFoundError(f"Drive 原始数据目录不存在：{drive_raw_src}")
    local_raw_dst = RAW_STAGE_ROOT / raw_folder
    local_raw_dst.parent.mkdir(parents=True, exist_ok=True)
    print(f"同步原始数据到本地高速盘（跳过已一致文件）：{drive_raw_src} -> {local_raw_dst}")
    subprocess.check_call(
        [
            "rsync",
            "-a",
            "--info=progress2",
            "--exclude",
            "_downloads/",
            f"{drive_raw_src}/",
            f"{local_raw_dst}/",
        ]
    )
    if link_target.is_symlink() or link_target.exists():
        if link_target.is_symlink():
            link_target.unlink()
        elif link_target.is_dir():
            shutil.rmtree(link_target)
        else:
            link_target.unlink()
    link_target.parent.mkdir(parents=True, exist_ok=True)
    os.symlink(str(local_raw_dst), str(link_target))
    print(f"已将软链接桥更新至本地 SSD: {link_target} -> {local_raw_dst}")

os.environ["BRAINDATA_ROOT"] = LINKS_DIR
os.environ["BRAINDECODING_DATA_DIR"] = LINKS_DIR
os.environ["BRAINDECODING_CACHE_DIR"] = str(DERIVED_CACHE_DIR)
DRIVE_RUNS_DIR = DRIVE_BASE / "runs"
DRIVE_RUNS_DIR.mkdir(parents=True, exist_ok=True)
os.environ["BRAINDECODING_RESULTS_DIR"] = str(DRIVE_RUNS_DIR)
os.environ["BRAINDECODING_OUTPUT_DIR"] = str(DRIVE_RUNS_DIR / "outputs")
os.environ["HF_HOME"] = str(Path(LOCAL_CACHE_DIR) / "huggingface")
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["PYTHONPATH"] = str(REPO_PATH)

print("环境变量已就绪：")
for key in (
    "BRAINDATA_ROOT",
    "BRAINDECODING_DATA_DIR",
    "BRAINDECODING_CACHE_DIR",
    "BRAINDECODING_RESULTS_DIR",
    "HF_HOME",
):
    print(f"  {key}={os.environ[key]}")
        

In [ ]:
from pathlib import Path

raw_root = Path(os.environ["BRAINDATA_ROOT"]) / DATASET_RAW_SUBDIR[DATASET]
if not raw_root.exists():
    raise FileNotFoundError(f"未在软链接桥找到数据集根目录：{raw_root}")

if DATASET == "broderick2018":
    missing_mats = []
    for s in range(1, 20):
        for r in range(1, 21):
            mat_p = raw_root / "Natural Speech/EEG" / f"Subject{s}" / f"Subject{s}_Run{r}.mat"
            if not mat_p.is_file():
                missing_mats.append(str(mat_p))
    missing_priv = [
        p
        for r in range(1, 21)
        for p in (raw_root / "private" / f"align{r}.json", raw_root / "private" / f"oldman_run{r}.txt")
        if not p.is_file()
    ]
    if missing_mats or missing_priv:
        raise RuntimeError(
            f"Broderick 2018 原始数据不完整：缺 .mat {len(missing_mats)} 个，"
            f"缺 private 文件 {len(missing_priv)} 个。请先运行 colab_broderick2018_download.ipynb。"
        )
    print(f"Broderick 2018 前置核查通过（380 .mat + 20 align + 20 oldman_run）：{raw_root}")
else:
    entries = list(raw_root.iterdir())
    if not entries:
        raise RuntimeError(f"数据集目录为空：{raw_root}")
    print(f"{DATASET} 前置核查通过（顶层条目数 {len(entries)}）：{raw_root}")
        

In [ ]:
import json
import subprocess
from pathlib import Path

cache_cmd = [str(VENV_PYTHON), "-m", "braindecoding", "cache", "--dataset", DATASET]
if SUBJECTS.strip():
    cache_cmd.extend(["--subjects", SUBJECTS.strip()])
elif int(NUM_SUBJECTS) > 0:
    cache_cmd.extend(["--num-subjects", str(int(NUM_SUBJECTS))])
if INCLUDE_TEST:
    cache_cmd.append("--include-test")
if FORCE_REBUILD:
    cache_cmd.append("--force")

print("执行离线特征缓存构建：", " ".join(cache_cmd))
subprocess.check_call(cache_cmd, cwd=str(REPO_PATH), env=os.environ.copy())

ds_derived = DERIVED_CACHE_DIR / DATASET
if not (ds_derived / "events/events.csv").is_file():
    raise FileNotFoundError(f"期望的缓存事件表缺失：{ds_derived / 'events/events.csv'}")
print(f"[OK] events/events.csv ({(ds_derived / 'events/events.csv').stat().st_size:,} bytes)")

if DATASET == "broderick2018":
    expected_files = [
        ds_derived / "events/manifest.json",
        ds_derived / "signals/manifest.json",
        ds_derived / "text/t5_large_layer_0_5/embeddings.npz",
        ds_derived / "text/t5_large_layer_0_5/embeddings.json",
        ds_derived / "events/manifests/vocabulary_N50.json",
    ]
    if not SUBJECTS.strip() and int(NUM_SUBJECTS) <= 0 and INCLUDE_TEST:
        expected_files.extend(
            [
                ds_derived / "text/t5_large_layer_0_5/manifest.json",
                ds_derived / "manifest.json",
            ]
        )
    for artifact in expected_files:
        if not artifact.is_file():
            raise FileNotFoundError(f"期望的 Broderick2018 核心产物缺失：{artifact}")
        print(f"[OK] {artifact.relative_to(DERIVED_CACHE_DIR)} ({artifact.stat().st_size:,} bytes)")
    if (ds_derived / "manifest.json").is_file():
        ds_manifest = json.loads((ds_derived / "manifest.json").read_text(encoding="utf-8"))
        print("manifest.json status:", ds_manifest.get("status"))
else:
    signals_root = ds_derived / "signals"
    if not signals_root.is_dir() or not any(signals_root.rglob("*.npy")):
        raise FileNotFoundError(f"期望的信号缓存目录缺少 .npy：{signals_root}")
    print(f"[OK] {signals_root.relative_to(DERIVED_CACHE_DIR)} (已生成 .npy 信号)")

In [ ]:
import subprocess
import tarfile
from pathlib import Path

pack_script = REPO_PATH / "scripts/cache_archive.py"
pack_cmd = [
    str(VENV_PYTHON),
    str(pack_script),
    "pack",
    "--cache-dir",
    str(DERIVED_CACHE_DIR),
    "--archive-dir",
    str(DRIVE_ARCHIVES_DIR),
    "--modality",
    DATASET,
]
print("开始打包云端单文件 tar 归档：", " ".join(pack_cmd))
subprocess.check_call(pack_cmd, cwd=str(REPO_PATH), env=os.environ.copy())

if not ARCHIVE_PATH.is_file():
    raise FileNotFoundError(f"归档文件未生成：{ARCHIVE_PATH}")

with tarfile.open(ARCHIVE_PATH, "r:") as tf:
    member_names = [m.name.replace("\\", "/") for m in tf.getmembers() if m.isfile()]

has_events = any(name.endswith("events.csv") for name in member_names)
has_signals = any("/signals/" in name for name in member_names)
has_text = any(name.endswith("embeddings.npz") or name.endswith(".npz") for name in member_names)
has_manifest = any(name.endswith(".json") for name in member_names)

print(
    f"云端归档校验通过：{ARCHIVE_PATH} | "
    f"大小 {ARCHIVE_PATH.stat().st_size / (1024**2):.1f} MB | 文件总数 {len(member_names)}"
)
print(
    f"结构核查: events.csv={has_events}, signals={has_signals}, "
    f"text_npz={has_text}, json_metadata={has_manifest}"
)
if not (has_events and has_signals and has_text and has_manifest):
    raise RuntimeError(f"归档 {ARCHIVE_PATH} 缺少关键缓存部件，请检查上方输出。")